# EU AI Ethics Assistant — Expert Agent with Groq, RAG and LangGraph


An expert conversational agent on **European AI ethics and regulation**, grounded in 4 official/academic sources:
1. EU AI Act (full legal text)
2. A Survey on Bias and Fairness in Machine Learning (Mehrabi et al.)
3. European Parliament study: The Ethics of AI — Issues and Initiatives
4. EU Ethics Guidelines for Trustworthy AI (AI HLEG)

**Pipeline:** PDF ingestion → chunking → local Sentence Transformers embeddings → ChromaDB → LangGraph agent (retrieve → generate) with conversation memory, answered by Groq.

> **⚠️ Note on LLM/embeddings:** this project originally used Google Gemini for both the LLM and embeddings, as specified in the assignment brief. Gemini's free-tier quota (1,000 embedding requests/day, 100 requests/minute) repeatedly blocked development — the exact failure this notebook hit before the migration — so the stack was deliberately switched to **Groq** (LLM) + **local Sentence Transformers** (embeddings), both genuinely free with no rate limits. The RAG architecture, LangGraph agent design, conversation memory, and system prompt are all unchanged from the original design. Full rationale in `DECISIONS.md`.

**By Nazmul Farooquee**

**uuid - 98e732f8-7758-4b2a-aa88-39323ebd9bef**

In [ ]:
# --- Imports & environment ---
import os
import sys

# Fail loudly if the wrong kernel is selected — this project's venv lives in
# ai_ethics/, and mixing it up with an unrelated project's kernel is the most
# common source of confusing ModuleNotFoundError's during setup.
print("Python executable:", sys.executable)
if "ai_ethics" not in sys.executable:
    raise RuntimeError(
        "Wrong kernel selected!\n"
        f"Current interpreter: {sys.executable}\n"
        "This notebook needs the project's own venv kernel "
        "('!!! USE THIS ONE - AI ETHICS ASSISTANT !!!').\n"
        "Fix: click the kernel name (top-right) -> Select Another Kernel -> Python Environments -> "
        "pick the interpreter under ai-ethics-assistant\\ai_ethics\\Scripts\\python.exe, "
        "then Ctrl+Shift+P -> 'Jupyter: Restart Kernel', then re-run this cell."
    )

from dotenv import load_dotenv

# Load GROQ_API_KEY from the .env file in the project root
load_dotenv("../.env")

# Fail fast if the key is missing (better than a cryptic error later)
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found — check your .env file"

# --- Central configuration (one place to tune everything) ---
DATA_DIR        = "../data"
CHROMA_DIR      = "../chroma_db"
COLLECTION_NAME = "ai_ethics_eu"

LLM_MODEL       = "llama-3.1-8b-instant"  # free, fast, no rate limits (Groq)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"      # local Sentence Transformers — no API, no quota

CHUNK_SIZE      = 1400   # characters per chunk
CHUNK_OVERLAP   = 150    # overlap so ideas aren't cut mid-sentence
TOP_K           = 4      # chunks retrieved per question

print("Configuration loaded ✔")

WHY (for DECISIONS.md): chunk_size 1400 with 150 overlap is sized for two reasons: (1) dense legal/academic text needs big enough chunks to preserve a full legal provision, and (2) it keeps chunk count reasonable for fast local embedding. Since embeddings run locally via Sentence Transformers, there's no daily API quota to worry about (unlike the Gemini embeddings this project started with). TOP_K=4 balances context richness vs. noise. All tunable from one cell.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import re

def clean_text(text: str) -> str:
    """Light cleaning for PDF-extracted text."""
    text = re.sub(r"\s+", " ", text)          # collapse whitespace/newlines
    text = re.sub(r"-\s+", "", text)          # fix hyphenated line breaks ("regu- lation")
    return text.strip()

# Map each file to a human-readable source name (used later in citations)
SOURCES = {
    "EU-AI-Act.pdf":                 "EU AI Act",
    "bias_fairness_survey.pdf":      "Bias & Fairness Survey (Mehrabi et al.)",
    "ethics_of_ai_study.pdf":        "EP Study: Ethics of AI",
    "trustworthy_ai_guidelines.pdf": "Ethics Guidelines for Trustworthy AI",
}

documents = []
for filename, source_name in SOURCES.items():
    path = os.path.join(DATA_DIR, filename)
    loader = PyPDFLoader(path)
    pages = loader.load()                     # one Document per page
    for page in pages:
        page.page_content = clean_text(page.page_content)
        page.metadata["source_name"] = source_name   # attach clean source label
    documents.extend(pages)
    print(f"Loaded {len(pages):>3} pages ← {source_name}")

print(f"\nTotal pages loaded: {len(documents)}")

WHY: metadata source_name lets the agent say which document an answer came from — a big credibility boost in the demo.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],  # split at natural boundaries first
)

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} pages")

# Sanity check — inspect one chunk
print("\n--- Sample chunk ---")
print("Source:", chunks[50].metadata["source_name"])
print(chunks[50].page_content[:400])

WHY: RecursiveCharacterTextSplitter tries paragraph → sentence → word boundaries in order, so chunks stay semantically coherent. Overlap ensures a definition that starts at a chunk edge isn't lost.

In [ ]:
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer


class LocalEmbeddings:
    """Local sentence-transformers embeddings wrapper for LangChain compatibility."""

    def __init__(self, model_name: str):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode(text, show_progress_bar=False).tolist()


embeddings = LocalEmbeddings(EMBEDDING_MODEL)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

print(f"Indexing {len(chunks)} chunks...")
vectorstore.add_documents(chunks)

print(f"\nIndexed {vectorstore._collection.count()} chunks in ChromaDB ✔")

NOTE: Local embedding runs in one shot — no batching or rate-limit backoff needed (that machinery was only required for the Gemini free-tier quota this project used before switching to local Sentence Transformers). Indexing ~250 pages (~750 chunks) takes about 13 seconds. Run it ONCE. On later notebook restarts, load the existing store instead:

In [ ]:
# --- Reload existing store (use this after the first run) ---
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
print(f"Loaded existing collection: {vectorstore._collection.count()} chunks")

TIP (rubric best practice): test first with ONE pdf / a few pages to validate the pipeline, then index everything.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

test_queries = [
    "What are high-risk AI systems under the EU AI Act?",
    "What types of bias exist in machine learning?",
    "What are the seven requirements for trustworthy AI?",
]

for q in test_queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    results = retriever.invoke(q)
    for i, doc in enumerate(results, 1):
        print(f"\n[{i}] Source: {doc.metadata['source_name']} (page {doc.metadata.get('page', '?')})")
        print(doc.page_content[:250], "...")

WHAT TO CHECK: each query should return chunks from the right document (AI Act question → AI Act chunks, bias question → survey chunks). If not, revisit chunk size. Write a short markdown cell after this documenting what you observed — that's "Excellent" tier documentation.

In [ ]:
SYSTEM_PROMPT = """You are an expert assistant on European AI ethics and regulation.

Your knowledge base consists of four authoritative sources:
1. The EU AI Act (official legal text)
2. "A Survey on Bias and Fairness in Machine Learning" (Mehrabi et al.)
3. The European Parliament study "The Ethics of Artificial Intelligence: Issues and Initiatives"
4. The EU "Ethics Guidelines for Trustworthy AI" (AI HLEG)

RULES:
1. Answer ONLY using the context provided below. Never invent facts, article numbers, or legal requirements.
2. If the context does not contain the answer, say clearly: "I don't have enough information in my knowledge base to answer that" — and suggest what the user could ask instead.
3. Mention which source your answer is based on (e.g., "According to the EU AI Act...").
4. Be clear and educational: your users may be developers, students, or policymakers with no legal background. Explain technical or legal terms briefly when you use them.
5. Be concise: 2–4 short paragraphs maximum, unless the user asks for more detail.
6. You are not a lawyer. For legal decisions, recommend consulting a qualified professional.

CONTEXT FROM KNOWLEDGE BASE:
{context}
"""

### System Prompt — Design Justification

- **Grounding rule (1–2):** forces RAG-only answers and an explicit "I don't know" path — prevents hallucinated article numbers, the most dangerous failure for a legal/ethics domain.
- **Source attribution (3):** users can verify claims; builds trust and demonstrates retrieval is working.
- **Audience adaptation (4):** the tool targets non-experts — the whole point is making ~250 pages of regulation accessible.
- **Brevity (5):** a chatbot that answers with walls of text defeats its purpose.
- **Legal disclaimer (6):** responsible-AI practice for a compliance-adjacent domain — fitting for a project *about* AI ethics.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# --- 1. Agent state ---
# `add_messages` automatically APPENDS new messages instead of overwriting —
# this is what gives the agent conversation memory across turns.
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    context: str          # retrieved chunks for the CURRENT question

# --- 2. LLM ---
llm = ChatGroq(model=LLM_MODEL, temperature=0.2)
# temperature 0.2 → factual, low-creativity answers (right for legal/ethics content)

# --- 3. Node: retrieve ---
def retrieve(state: AgentState) -> dict:
    """Take the latest user question, fetch top-k chunks from ChromaDB."""
    question = state["messages"][-1].content
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(
        f"[Source: {d.metadata['source_name']}]\n{d.page_content}" for d in docs
    )
    return {"context": context}

# --- 4. Node: generate ---
def generate(state: AgentState) -> dict:
    """Build the prompt (system + context + full history) and call Groq."""
    system = SystemMessage(content=SYSTEM_PROMPT.format(context=state["context"]))
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}   # add_messages appends this to history

# --- 5. Build the graph:  START → retrieve → generate → END ---
graph_builder = StateGraph(AgentState)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

# --- 6. Compile with a checkpointer = conversation memory ---
memory = MemorySaver()
agent = graph_builder.compile(checkpointer=memory)

print("Agent compiled ✔")

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

HOW MEMORY WORKS (say this in the demo): MemorySaver checkpoints the full state after every turn, keyed by a thread_id. Same thread_id → the agent sees the whole message history. New thread_id → fresh conversation. The add_messages reducer is what accumulates history instead of overwriting it.

In [ ]:
import time
from groq import RateLimitError

def ask(question: str, thread_id: str = "demo", max_retries: int = 3) -> str:
    """Send one question to the agent within a persistent conversation thread.

    Groq's free tier caps tokens-per-minute, which the back-to-back demo
    calls below can hit; retry with a short backoff instead of failing the cell.
    """
    config = {"configurable": {"thread_id": thread_id}}
    for attempt in range(max_retries):
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=question)]},
                config=config,
            )
            break
        except RateLimitError:
            wait_s = 15
            print(f"  Rate limit hit, retrying in {wait_s}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait_s)
    else:
        raise RuntimeError(f"Failed to get a response after {max_retries} retries")

    answer = result["messages"][-1].content
    print(f"🧑 Q: {question}\n")
    print(f"🤖 A: {answer}")
    print("\n" + "=" * 70)
    return answer

In [ ]:
# Example 1 — Regulation (should cite the EU AI Act)
ask("What is considered a high-risk AI system under the EU AI Act?")

In [ ]:
# Example 2 — Technical foundations (should cite the Bias & Fairness survey)
ask("What are the main types of bias that can appear in machine learning systems?")

In [ ]:
# Example 3 — Ethics principles (should cite the Trustworthy AI Guidelines)
ask("What are the seven key requirements for trustworthy AI in the EU?")

In [ ]:
# Example 4 — Real-world impact (should cite the EP Ethics study)
ask("What ethical concerns does AI raise in healthcare?")

In [ ]:
# Example 5 — OUT-OF-SCOPE test: the agent should say "I don't know" ⭐
ask("What does Japanese AI regulation say about facial recognition?")

In [ ]:
# Turn 1
ask("What is algorithmic bias?", thread_id="memory-demo")

In [ ]:
# Turn 2 — refers to "it" with no other clue: only works if memory works
ask("Can you give me two real-world examples of it?", thread_id="memory-demo")

In [ ]:
# Turn 3 — refers to the whole previous exchange
ask("Which of those two examples is covered by the EU AI Act?", thread_id="memory-demo")

In [ ]:
print("💬 AI Ethics Assistant — type your question ('exit' to quit)\n")

while True:
    try:
        question = input("You: ").strip()
    except Exception:
        # No interactive stdin available (e.g. batch/CI execution, or a
        # headless kernel that raises StdinNotImplementedError instead of
        # EOFError) — stop cleanly rather than crashing the cell.
        print("(no interactive input available — skipping chat loop)")
        break
    if question.lower() in {"exit", "quit", "salir"}:
        print("Goodbye! 👋")
        break
    if question:
        ask(question, thread_id="interactive")